In [ ]:
%load_ext autoreload
%autoreload 2
from hypnose_behavior.trial_classification.run import *
from hypnose_behavior.io.detect_stage import detect_stage
from hypnose_behavior.io.loaders import load_experiment
from hypnose_behavior.visualization.valve_poke_plots import plot_valve_and_poke_events
from ipywidgets import widgets
from IPython.display import display

%matplotlib widget

In [ ]:
# Multi Date or SubjID Analysis. Can analyze all Sessions for given SubjID (run on subjid only), all SubjIDs for a date (run on date only), or specific SubjID(s) and Date(s) (run on lists of each)
# To analyze all subjids for a date, or vice versa, set the other argument to None
# For Dates: use lists for specific dates [YYYYMMDD], or use range(start_date, end_date) for a date range (inclusive)

subjids = [61]
dates = [20260908]

multi_run_results = batch_analyze_sessions(subjids=subjids, dates=dates, save=True, verbose=False, print_summary=True)

In [ ]:
# Plot the raw valve and poke events for a given session. Optionally define a time window to only plot the specific time range of interest. 

root = load_experiment(57, 20260806, index=0)
time_window = ('15:46:30', '15:47:30')
plot_valve_activity = plot_valve_and_poke_events(root=root, time_window=None)

In [ ]:
res = analyze_session_multi_run_by_id_date(57, 20260529, verbose=False, print_summary=True, save=True)

In [ ]:
# Loading experiments --> just define the SUBJID and DATE
root = load_experiment(45, 20260223, index=1) #can add index for multiple experiments; index=0 as default
stage = detect_stage(root)

# Miscellaneous 

In [ ]:
# Find X s window with the most rewarded trials - used to find video segments
window_sec = 60

def find_peak_rewarded_window(res, window_sec=40):
    # Get rewarded trials table
    cls = res.get("classification", res)
    df = cls.get("completed_sequence_rewarded", pd.DataFrame())
    if df.empty:
        print("No rewarded trials found.")
        return None

    # Use valve_open_ts as trial time (or poke_first_in if you prefer)
    times = pd.to_datetime(df["sequence_start"], errors="coerce")
    df = df.assign(trial_time=times)
    df = df.dropna(subset=["trial_time"]).sort_values("trial_time").reset_index(drop=True)

    # Add relative time from start (seconds and HH:MM:SS:MS) to align with video timeline
    start_time = df["trial_time"].min()
    deltas = df["trial_time"] - start_time
    rel_ms = deltas.dt.total_seconds() * 1000.0

    def _fmt_ms(ms):
        if pd.isna(ms):
            return None
        ms_int = int(round(ms))
        hours, rem = divmod(ms_int, 3600_000)
        minutes, rem = divmod(rem, 60_000)
        seconds, millis = divmod(rem, 1000)
        return f"{hours:02d}:{minutes:02d}:{seconds:02d}:{millis:03d}"

    df = df.assign(
        trial_time_rel_sec=rel_ms / 1000.0,
        trial_time_rel_hmsms=rel_ms.apply(_fmt_ms),
    )

    # Find the window with the most rewarded trials
    best_count = 0
    best_start = None
    best_end = None
    best_indices = []

    trial_times = df["trial_time"].values
    n = len(trial_times)
    for i in range(n):
        start = trial_times[i]
        end = start + np.timedelta64(window_sec, "s")
        # Find all trials within [start, end)
        mask = (trial_times >= start) & (trial_times < end)
        count = mask.sum()
        if count > best_count:
            best_count = count
            best_start = start
            best_end = end
            best_indices = np.where(mask)[0]

    print(f"Max rewarded trials in any {window_sec}s window: {best_count}")
    print(f"Window: {best_start} to {best_end}")
    # Optionally display the trials in that window (includes trial_time_rel_sec and HH:MM:SS:MS)
    display(df.iloc[best_indices])
    return df.iloc[best_indices]

peak_window_trials = find_peak_rewarded_window(res, window_sec=window_sec)


# Debugging Functions:

